# 8-1절 연습 문제 풀이

이 노트북은 8-1절 연습 문제의 풀이 예시다. 정답이 하나뿐인 문제가 아니므로 다른 구현도 얼마든지 가능하다.

- 본문 예제 코드는 `notebooks/ch08/` 아래 예제 노트북을 참고한다.
- 위에서부터 차례대로 실행한다.

In [ ]:
# 환경 설정 - 공통 라이브러리, 시드 고정, 장치 객체
import sys
sys.path.append('../../')

import random

import numpy as np
import torch
import torch.nn as nn

from code_reference import common
# viz.configure()에서 save_grayscale=True로 지정하면 노트북에 표시되는 시각화 이미지를 파일로 저장함
from code_reference import visualize as viz

viz.configure(save_grayscale=False)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = common.get_device()

# 8장 공통 - MiniVGGNet / 배치 정규화 / 잔차 블록
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
import torchinfo
DATA_ROOT = '../../download'

def cifar_loaders(batch_size=64, train_transform=None):
    tf = transforms.Compose([transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))])
    full = datasets.CIFAR10(root=DATA_ROOT, train=True, download=True,
                            transform=train_transform or tf)
    test = datasets.CIFAR10(root=DATA_ROOT, train=False, download=True, transform=tf)
    g = torch.Generator().manual_seed(SEED)
    n_val = int(len(full) * 0.2)
    tr, va = random_split(full, [len(full) - n_val, n_val], generator=g)
    return (DataLoader(tr, batch_size=batch_size, shuffle=True),
            DataLoader(va, batch_size=batch_size), DataLoader(test, batch_size=batch_size))

def vgg_block(fan_in, fan_out, n_conv=2, bn=False):
    layers = []
    for i in range(n_conv):
        layers.append(nn.Conv2d(fan_in if i == 0 else fan_out, fan_out, 3, 1, 1))
        if bn: layers.append(nn.BatchNorm2d(fan_out))
        layers.append(nn.ReLU())
    layers.append(nn.MaxPool2d(2))
    return nn.Sequential(*layers)

def make_vgg(channels=(32, 64, 128), n_conv=2, bn=False, dropout=0.0):
    blocks, fan_in, size = [], 3, 32
    for ch in channels:
        blocks.append(vgg_block(fan_in, ch, n_conv, bn)); fan_in, size = ch, size // 2
    head = [nn.Flatten()]
    if dropout: head.append(nn.Dropout(dropout))
    head.append(nn.Linear(fan_in * size * size, 10))
    return nn.Sequential(*blocks, *head)

def fit(model, epochs=10, lr=1e-3, loaders=None):
    tr, va, te = loaders or cifar_loaders()
    model = model.to(device)
    crit = nn.CrossEntropyLoss()
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for e in range(1, epochs + 1):
        model.train()
        for x, y in tr:
            loss = crit(model(x.to(device)), y.to(device))
            opt.zero_grad(); loss.backward(); opt.step()
        model.eval(); c = n = 0
        with torch.no_grad():
            for x, y in va:
                c += (model(x.to(device)).argmax(1).cpu() == y).sum().item(); n += len(y)
        print(f'  {e}/{epochs} 검증 정확도 {c / n * 100:.2f}%')
    model.eval(); c = n = 0
    with torch.no_grad():
        for x, y in te:
            c += (model(x.to(device)).argmax(1).cpu() == y).sum().item(); n += len(y)
    print(f'  평가 정확도 {c / n * 100:.2f}%')
    return c / n * 100

## 연습 8-1

MiniVGGNet 모델은 합성곱 계층이 출력하는 데이터의 채널 수를 32, 64, 128로 두 배씩 늘려 가며 특징을 추출한다. 이 모델의 모든 합성곱 계층의 출력 채널 수를 64로 고정한 후, 다음 관점으로 결과를 확인해 보자.

전체 파라미터 수와 메모리 사용량의 변화

모델의 성능

In [ ]:
for name, ch in [('32-64-128 (원본)', (32, 64, 128)), ('64 고정', (64, 64, 64))]:
    torch.manual_seed(SEED)
    model = make_vgg(ch)
    n_param = sum(p.numel() for p in model.parameters())
    print(f'[{name}] 파라미터 {n_param:,}개')
    fit(model, epochs=10)
    print()

채널을 64로 고정하면 앞쪽 블록의 파라미터는 늘고 뒤쪽은 줄어, 전체 파라미터는 오히려 **감소**한다. 특징 지도가 작아지는 뒤쪽에서 채널을 늘리는 편이 효율적이기 때문이다.

성능도 대체로 원본이 낫다. 깊이가 깊어질수록 **추상적인 특징의 종류가 늘어나므로** 채널을 함께 늘리는 것이 VGGNet의 설계 원칙이다.

## 연습 8-2

VGGNet이 입증한 것처럼 모델의 깊이를 늘릴수록 모델 성능이 좋아지는지 MiniVGGNet 모델을 다음 두 가지 방향으로 각각 수정해 직접 확인해 보자.

VGG 블록에 합성곱 계층과 ReLU 활성화 계층 한 쌍을 더 추가한 모델

특징 추출기 계층(features)에 VGG 블록을 하나 더 추가한 모델

계층을 추가했음에도 불구하고 모델의 성능이 좋아지지 않았다면 그 이유가 무엇일지 추측해 보자.

In [ ]:
configs = [('원본 (블록당 합성곱 2개)', dict(channels=(32, 64, 128), n_conv=2)),
           ('블록당 합성곱 3개', dict(channels=(32, 64, 128), n_conv=3)),
           ('VGG 블록 4개', dict(channels=(32, 64, 128, 256), n_conv=2))]
for name, kw in configs:
    torch.manual_seed(SEED)
    model = make_vgg(**kw)
    print(f'[{name}] 파라미터 {sum(p.numel() for p in model.parameters()):,}개')
    fit(model, epochs=10)
    print()

깊이를 늘려도 성능이 계속 좋아지지는 않는다. 배치 정규화 없이 층만 쌓으면 **기울기가 앞쪽 층까지 잘 전달되지 않아** 학습이 불안정해진다. VGG 블록 4개 구성은 특징 지도가 2×2까지 줄어 공간 정보도 부족해진다.

이 한계를 푸는 것이 8-2절의 배치 정규화와 8-3절의 잔차 연결이다.

## 연습 8-3

[도전 문제] 완전 합성곱 오토인코더를 만드는 [연습 문제 7-12]의 결과에 인코더와 디코더 모두 VGGNet의 아이디어를 이식해 보면 어떨까? 다음 과정으로 그 아이디어를 담은 완전 합성곱 오토인코더 모델을 만들어 보자.

CIFAR-10 데이터셋으로 학습 및 평가가 가능하도록 데이터셋, 데이터로더, 모델을 수정한다.

인코더와 디코더가 VGGNet처럼 3x3 필터만 사용하도록 수정하고, 합성곱 계층을 더 추가한다.

CIFAR-10 데이터셋을 사용해 오토인코더 모델을 학습한 후 이미지 복원 결과를 비교한다.

In [ ]:
# VGG 아이디어를 담은 완전 합성곱 오토인코더 (CIFAR-10)
class VGGAutoEncoder(nn.Module):
    def __init__(self, bn=False):
        super().__init__()
        def enc_block(i, o):
            layers = [nn.Conv2d(i, o, 3, 1, 1)]
            if bn: layers.append(nn.BatchNorm2d(o))
            layers += [nn.ReLU(), nn.Conv2d(o, o, 3, 1, 1)]
            if bn: layers.append(nn.BatchNorm2d(o))
            layers += [nn.ReLU(), nn.MaxPool2d(2)]
            return nn.Sequential(*layers)
        def dec_block(i, o):
            layers = [nn.ConvTranspose2d(i, o, 3, 2, 1, output_padding=1)]
            if bn: layers.append(nn.BatchNorm2d(o))
            layers += [nn.ReLU(), nn.Conv2d(o, o, 3, 1, 1)]
            if bn: layers.append(nn.BatchNorm2d(o))
            layers.append(nn.ReLU())
            return nn.Sequential(*layers)
        self.encoder = nn.Sequential(enc_block(3, 32), enc_block(32, 64))   # 32->8
        self.decoder = nn.Sequential(dec_block(64, 32), dec_block(32, 16),
                                     nn.Conv2d(16, 3, 3, 1, 1), nn.Sigmoid())
    def forward(self, x): return self.decoder(self.encoder(x))

x = torch.randn(2, 3, 32, 32)
m = VGGAutoEncoder()
print(f'잠재(특징 지도): {tuple(m.encoder(x).shape)} / 복원: {tuple(m(x).shape)}')

In [ ]:
tf = transforms.ToTensor()      # 오토인코더는 0~1 범위 그대로 사용
train_set = datasets.CIFAR10(root=DATA_ROOT, train=True, download=True, transform=tf)
loader = DataLoader(train_set, batch_size=128, shuffle=True)
torch.manual_seed(SEED)
ae = VGGAutoEncoder().to(device)
crit = nn.MSELoss(); opt = torch.optim.Adam(ae.parameters(), lr=1e-3)
for e in range(1, 6):
    ae.train(); tot = n = 0
    for x, _ in loader:
        x = x.to(device)
        loss = crit(ae(x), x)
        opt.zero_grad(); loss.backward(); opt.step()
        tot += loss.item() * len(x); n += len(x)
    print(f'{e}/5 복원 손실 {tot / n:.5f}')

test_x = torch.stack([datasets.CIFAR10(root=DATA_ROOT, train=False,
                      transform=tf)[i][0] for i in range(8)]).to(device)
ae.eval()
with torch.no_grad():
    recon = ae(test_x)
viz.plot_images(list(test_x.cpu()) + list(recon.cpu()),
                ['원본'] * 8 + ['복원'] * 8, images_per_row=8)

인코더는 VGG 블록(3×3 합성곱 2개 + 풀링)으로 채널을 늘리며 크기를 줄이고, 디코더는 역합성곱으로 대칭적으로 되돌린다. 선형 계층이 전혀 없는 **완전 합성곱** 구조라 입력 크기가 달라져도 동작한다.